# Laura++ integration validation with the E791 Fit 2 model

This notebook compares the Laura++-style Gauss--Legendre quadrature with the equal-area Dalitz grid using the seven-component E791 $D^+\to\pi^-\pi^+\pi^+$ Fit 2 model already used in this project. It checks the Dalitz area, every complex normalization-matrix element, the coherent normalization, and direct-PDF versus cached-matrix closure.

The Laura++ base method uses Gauss--Legendre abscissas in $m_{13}$ and $m_{23}$ with the Jacobian $4m_{13}m_{23}$. The special narrow-resonance sub-grid extension is not part of this test.

In [ ]:
import time

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from dalitzplotfitter import (
    DalitzGrid, DecayChannel, DecayModel, LauraGaussLegendreGrid,
    NonResonant, RealImag, Resonance, enable_x64,
)
from dalitzplotfitter.integration import matrix_normalization, normalization_matrix

enable_x64()

## E791 Fit 2 amplitude model

The published polar coefficients and the project convention for the nonresonant phase are kept identical to `01_e791_dplus_fit2_generation.ipynb`.

In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma": (1.17, 205.7), "rho770": (1.0, 0.0),
    "NR": (0.48, 57.3), "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3), "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(name):
    magnitude, phase_deg = fit2_polar[name]
    if name == "NR":
        phase_deg += 180.0
    phase = np.deg2rad(phase_deg)
    return magnitude * np.cos(phase), magnitude * np.sin(phase)

def coefficient(name):
    return RealImag(*polar_to_xy(name))

components = [
    Resonance("sigma", (0, 1), coefficient("sigma"), mass=0.478, width=0.324, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0, 1), coefficient("rho770"), mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0, 1), coefficient("f0_980"), mass=0.975, width=0.044, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0, 1), coefficient("f2_1270"), mass=1.275, width=0.185, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0, 1), coefficient("f0_1370"), mass=1.434, width=0.173, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0, 1), coefficient("rho1450"), mass=1.465, width=0.310, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficient("NR")),
]

model = DecayModel(channel, components, normalize_components=False, normalization_resolution=80)
names = [component.name for component in model.amplitude_model.components]
coefficients = jnp.asarray([component.coefficient.value({}) for component in model.amplitude_model.components])
print(channel)
print(names)

## Matrix helpers

Both methods are evaluated with the same amplitude functions. The raw matrix is $M_{ij}=\int F_i^*F_j\,d\Phi$. The unit-diagonal matrix applies the package's individual-component normalization to the same raw result.

In [ ]:
def raw_matrix(sample):
    values = jnp.stack([
        component.function(sample.as_dict(), None)
        for component in model.amplitude_model.components
    ], axis=1)
    return normalization_matrix(values, sample.weights)

def unit_diagonal_matrix(matrix):
    scale = 1.0 / jnp.sqrt(jnp.real(jnp.diag(matrix)))
    return jnp.conj(scale)[:, None] * matrix * scale[None, :]

def metrics(matrix, reference):
    delta = matrix - reference
    raw_frobenius = float(jnp.linalg.norm(delta) / jnp.linalg.norm(reference))
    unit = unit_diagonal_matrix(matrix)
    unit_reference = unit_diagonal_matrix(reference)
    unit_delta = unit - unit_reference
    unit_frobenius = float(jnp.linalg.norm(unit_delta) / jnp.linalg.norm(unit_reference))
    maximum_unit_absolute = float(jnp.max(jnp.abs(unit_delta)))
    norm = float(matrix_normalization(coefficients, matrix))
    ref_norm = float(matrix_normalization(coefficients, reference))
    return raw_frobenius, unit_frobenius, maximum_unit_absolute, abs(norm / ref_norm - 1.0), norm

def evaluate(sample):
    start = time.perf_counter()
    matrix = raw_matrix(sample)
    jnp.asarray(matrix).block_until_ready()
    return matrix, time.perf_counter() - start

## Dense equal-area reference and Laura++ default

The reference is intentionally independent of the new integrator. Increase `REFERENCE_N` for final production studies if memory permits.

In [ ]:
REFERENCE_N = 600
reference_grid = DalitzGrid(channel.parent_mass, channel.daughter_masses, resolution=REFERENCE_N)
reference_sample = reference_grid.sample()
reference_matrix, reference_time = evaluate(reference_sample)

laura_grid = LauraGaussLegendreGrid(channel.parent_mass, channel.daughter_masses, bin_width=0.005)
laura_sample = laura_grid.sample()
laura_matrix, laura_time = evaluate(laura_sample)

area_reference = float(reference_grid.area)
area_laura = float(jnp.mean(laura_sample.weights))
print(f"equal-area reference: {reference_sample.size:,} points, {reference_time:.3f} s")
print(f"Laura++ default    : {laura_sample.size:,} physical points, orders={laura_grid.orders}, {laura_time:.3f} s")
print(f"area relative difference: {abs(area_laura / area_reference - 1.0):.3e}")
print("metrics (raw Frobenius, unit Frobenius, max unit absolute, coherent norm, norm):")
print(metrics(laura_matrix, reference_matrix))

## Convergence scan

The explicit Laura++ orders are scanned on both mass axes. Errors are evaluated for the complete complex matrix, not only its diagonal or one coherent coefficient choice.

In [ ]:
orders = [60, 90, 130, 180, 240, 320]
rows = []
for order in orders:
    grid = LauraGaussLegendreGrid(
        channel.parent_mass, channel.daughter_masses,
        order_m13=order, order_m23=order,
    )
    sample = grid.sample()
    matrix, elapsed = evaluate(sample)
    raw_frob, unit_frob, max_unit_abs, norm_error, norm = metrics(matrix, reference_matrix)
    area_error = abs(float(jnp.mean(sample.weights)) / area_reference - 1.0)
    rows.append((order, sample.size, area_error, raw_frob, unit_frob, max_unit_abs, norm_error, elapsed))

print(" order    points    area rel.    raw Fro.    unit Fro.    max unit abs.    norm rel.    seconds")
for row in rows:
    print(f"{row[0]:6d} {row[1]:9d} {row[2]:12.3e} {row[3]:12.3e} {row[4]:12.3e} {row[5]:15.3e} {row[6]:12.3e} {row[7]:10.3f}")

In [ ]:
orders_plot = np.asarray([row[0] for row in rows])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].loglog(orders_plot, [row[2] for row in rows], "o-", label="Dalitz area")
axes[0].loglog(orders_plot, [row[3] for row in rows], "s-", label="raw matrix (Frobenius)")
axes[0].loglog(orders_plot, [row[4] for row in rows], "d-", label="unit-diagonal matrix (Frobenius)")
axes[0].loglog(orders_plot, [row[6] for row in rows], "^-", label=r"$c^\dagger M c$")
axes[0].set(xlabel="Gauss--Legendre order per axis", ylabel="relative difference", title="Convergence against equal-area reference")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].legend()

unit_difference = np.abs(np.asarray(unit_diagonal_matrix(laura_matrix) - unit_diagonal_matrix(reference_matrix)))
image = axes[1].imshow(np.log10(np.maximum(unit_difference, 1.0e-16)), vmin=-8, vmax=-1, cmap="viridis")
axes[1].set_xticks(range(len(names)), names, rotation=45, ha="right")
axes[1].set_yticks(range(len(names)), names)
axes[1].set_title(r"$\log_{10}$ absolute difference, unit-diagonal $M_{ij}$")
fig.colorbar(image, ax=axes[1])
plt.show()

## Direct PDF versus cached matrix closure

This final check uses `DecayModel(normalization_method="laura")` exactly as a fit would. Individual component normalization is enabled, and the direct intensity integral must equal the cache result on the identical quadrature sample.

In [ ]:
laura_model = DecayModel(
    channel, components,
    normalize_components=True,
    normalization_method="laura",
    normalization_bin_width=0.005,
)
data = laura_model.generate_phase_space(2048, seed=791)
cache = laura_model.prepare_cache(data)
sample = laura_model.normalization_sample
direct_norm = jnp.mean(sample.weights * laura_model.intensity(sample.as_dict(), {}))
cached_norm = cache.normalization({})
matrix = cache.normalization_matrix_fixed

print(f"direct normalization: {float(direct_norm):.15g}")
print(f"cached normalization: {float(cached_norm):.15g}")
print(f"relative closure    : {float(jnp.abs(direct_norm / cached_norm - 1.0)):.3e}")
print("unit diagonal      :", np.asarray(jnp.real(jnp.diag(matrix))))

assert jnp.allclose(direct_norm, cached_norm, rtol=2.0e-12, atol=2.0e-12)
assert jnp.allclose(jnp.real(jnp.diag(matrix)), jnp.ones(len(names)), rtol=2.0e-12, atol=2.0e-12)

## Interpretation

For acceptance, inspect the convergence plateau of the complete matrix and the largest relevant $M_{ij}$ difference. Agreement of only the total normalization can conceal compensating errors between diagonal and interference terms. For a production choice, repeat with a denser independent reference and record the tolerance required by the analysis.